In [1]:
import cv2
import mediapipe as mp

In [2]:
class HandDetector:
    def __init__(self):
        self.model_path = "hand_landmarker.task"
        self.BaseOptions = mp.tasks.BaseOptions(model_asset_path=self.model_path)
        self.options = mp.tasks.vision.HandLandmarkerOptions(
            base_options=self.BaseOptions,
            running_mode=mp.tasks.vision.RunningMode.VIDEO,num_hands=2)
        self.hand = mp.tasks.vision.HandLandmarker.create_from_options(self.options)
        self.timestamp_ms = 0

    def find_hand(self, image):
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image)
        self.detection_result = self.hand.detect_for_video(mp_image, self.timestamp_ms)
        self.timestamp_ms += 33
        for hand_landmarks in self.detection_result.hand_landmarks:
            mp.tasks.vision.drawing_utils.draw_landmarks(image, hand_landmarks,
                                                          mp.tasks.vision.HandLandmarksConnections.HAND_CONNECTIONS)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        return image

    def find_position(self, image):
        h, w, c = image.shape
        lm_list = []
        if self.detection_result.hand_landmarks:
            for hand_landmarks in self.detection_result.hand_landmarks:
                for id, lm in enumerate(hand_landmarks):
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    lm_list.append((id, cx, cy))
        return lm_list

In [10]:
import time
import math
import numpy as np
import subprocess

In [12]:
pTime = 0
cTime = 0
cap = cv2.VideoCapture(0)
detector = HandDetector()
while True:
    success, img = cap.read()
    img = detector.find_hand(img)
    lmList = detector.find_position(img)
    if len(lmList) != 0:
        x1, y1 = lmList[4][1], lmList[4][2]
        x2, y2 = lmList[8][1], lmList[8][2]
        length = math.hypot(x2 - x1, y2 - y1)
        vol = np.interp(length, [20, 200], [0, 100])
        subprocess.run(["wpctl", "set-volume", "58", f"{vol / 100:.2f}"])
        cv2.putText(img, str(int(length)), (x1, y1 - 20),
            cv2.FONT_HERSHEY_PLAIN, 2, (255, 0, 0), 2)
        cv2.putText(img, f"Volume: {int(vol)}", (10, 100),
            cv2.FONT_HERSHEY_PLAIN, 2, (255, 0, 0), 2)
        cv2.circle(img, (x1, y1), 15, (255, 0, 0), cv2.FILLED)
        cv2.circle(img, (x2, y2), 15, (255, 0, 0), cv2.FILLED)
        cv2.line(img, (x1, y1), (x2, y2), (255, 0, 0), 3)
        
    cTime = cv2.getTickCount()
    fps = cv2.getTickFrequency() / (cTime - pTime)
    pTime = cTime

    cv2.putText(img, str(int(fps)), (10, 70), cv2.FONT_HERSHEY_PLAIN, 3,
                (255, 0, 255), 3)

    cv2.imshow("Image", img)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

W0000 00:00:1788873683.554334   16357 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1788873683.565689   16357 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
